# Dense spatial P2–P5 fusion — Home Fire + D-Fire, 60 epochs, seeds 42 and 2026

Trains exactly **two proposed nano models**, one seed per T4. No baseline is trained.
Images are shuffled normally; this model has **no previous-frame memory**.

1. Import this notebook into a fresh Kaggle session. Select **GPU T4 x2** and enable Internet.
2. Attach a private Dataset made from **home_fire_dfire_20261102.zip**.
3. Attach a second private Dataset made from **dense_spatial_home_dfire_assets.zip**.
4. Run All. Download the linked `*_results.zip`, including partial results if a seed fails.

All images and complete labels are included in the combined data. Original source splits are kept.
D-Fire IDs are converted to fire=0/smoke=1. Established Home Fire orientation corrections are applied.
Validation remains exploratory because source splits may share scenes. Test is never used by this notebook.
These two runs do not establish improvement over a baseline, especially old Home Fire-only scores.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os, sys, json
DATASET_ROOT = None  # Optional exact combined-dataset folder or ZIP path.
ASSET_ROOT = None    # Optional exact companion folder or ZIP path.
SEEDS = [42, 2026]
GPU_IDS = [0, 1]
EPOCHS, IMGSZ, BATCH, WORKERS = 60, 640, 8, 2
assert (SEEDS, GPU_IDS, EPOCHS, IMGSZ, BATCH, WORKERS) == ([42,2026],[0,1],60,640,8,2)
RUN_ROOT = Path('/kaggle/working') / ('dense_spatial_home_dfire_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f'))
RUN_ROOT.mkdir(parents=True, exist_ok=False)
(RUN_ROOT/'settings').mkdir()
os.environ.update(YOLO_CONFIG_DIR=str(RUN_ROOT/'settings'), YOLO_AUTOINSTALL='false',
    WANDB_MODE='disabled', COMET_MODE='DISABLED', CUBLAS_WORKSPACE_CONFIG=':4096:8')
print('Output:',RUN_ROOT)


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, zipfile

EXPECTED_MANIFEST_SHA256 = '34b072689f0393a3b387d0b3a72420745a6d3ec19adeed7e0c8466d440177195'
EXPECTED_ZIP_SHA256 = '568cbe00e938a12ce7dc253507b9d31d292eba5e0cd07b13c1e5ecd50a6c8c26'
BUNDLE_FILENAME = 'dense_spatial_home_dfire_assets.zip'
MANIFEST_FILENAME = 'challenger-bundle.json'

def bundle_hash(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024*1024), b''):
            digest.update(block)
    return digest.hexdigest()

def locate_bundle(search_root, explicit=None):
    if explicit:
        path = Path(explicit)
        if not path.exists():
            raise FileNotFoundError('ASSET_ROOT does not exist: '+str(path))
        return path
    candidates = []
    visited = set()
    for current, directories, files in os.walk(search_root, followlinks=True):
        folder = Path(current)
        if folder.resolve() in visited:
            directories[:] = []
            continue
        visited.add(folder.resolve())
        if MANIFEST_FILENAME in files:
            if bundle_hash(folder/MANIFEST_FILENAME)==EXPECTED_MANIFEST_SHA256:
                candidates.append(folder)
            directories[:] = []
            continue
        if BUNDLE_FILENAME in files:
            path = folder/BUNDLE_FILENAME
            if bundle_hash(path)==EXPECTED_ZIP_SHA256:
                candidates.append(path)
        directories[:] = [name for name in directories if name.lower() not in
            {'images','labels','test','testing','.git','__pycache__'}]
    if not candidates:
        raise RuntimeError('Attach a Kaggle Dataset created from dense_spatial_home_dfire_assets.zip. '
            'Both the ZIP file and Kaggle-auto-extracted folders are supported. '
            'Alternatively set ASSET_ROOT to the exact ZIP/folder. No matching verified bundle was found.')
    # Duplicate copies have the same immutable identity; prefer an extracted copy.
    return sorted(candidates, key=lambda p:(not p.is_dir(),str(p)))[0]

def materialize_bundle(source, destination):
    source, destination = Path(source), Path(destination)
    if source.is_file():
        if bundle_hash(source)!=EXPECTED_ZIP_SHA256:
            raise RuntimeError('Companion ZIP hash mismatch; use the ZIP supplied with this notebook.')
        destination.mkdir(parents=True, exist_ok=False)
        with zipfile.ZipFile(source) as archive:
            if len(archive.namelist())!=len(set(archive.namelist())):
                raise RuntimeError('Duplicate bundle archive entries')
            for name in archive.namelist():
                if not (destination/name).resolve().is_relative_to(destination.resolve()):
                    raise RuntimeError('Invalid companion archive path')
            archive.extractall(destination)
    elif source.is_dir():
        if bundle_hash(source/MANIFEST_FILENAME)!=EXPECTED_MANIFEST_SHA256:
            raise RuntimeError('Companion folder manifest mismatch')
        manifest = json.loads((source/MANIFEST_FILENAME).read_text())
        destination.mkdir(parents=True, exist_ok=False)
        for name, expected in manifest['files'].items():
            target = destination/name
            if not target.resolve().is_relative_to(destination.resolve()):
                raise RuntimeError('Invalid companion manifest path')
            if bundle_hash(source/name)!=expected:
                raise RuntimeError('Companion file hash mismatch: '+name)
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source/name,target)
        shutil.copy2(source/MANIFEST_FILENAME,destination/MANIFEST_FILENAME)
    else:
        raise RuntimeError('ASSET_ROOT must be a ZIP file or extracted folder')
    if bundle_hash(destination/MANIFEST_FILENAME)!=EXPECTED_MANIFEST_SHA256:
        raise RuntimeError('Companion manifest identity mismatch')
    manifest=json.loads((destination/MANIFEST_FILENAME).read_text())
    for name, expected in manifest['files'].items():
        if bundle_hash(destination/name)!=expected:
            raise RuntimeError('Extracted companion file hash mismatch: '+name)
    return destination

# Load attached assets. This cell does not import Torch or start training.
BUNDLE = locate_bundle(Path('/kaggle/input'), ASSET_ROOT)
ASSETS = materialize_bundle(BUNDLE, RUN_ROOT/'assets')
print('Verified companion assets:', BUNDLE)


In [ ]:
# The previous cell verified and copied the attached assets into ASSETS.
if any(name=='ultralytics' or name.startswith('ultralytics.') for name in sys.modules):
    raise RuntimeError('Restart the session, then run this notebook from the top.')
VENDOR = RUN_ROOT/'vendor'
with zipfile.ZipFile(ASSETS/'ultralytics-8.4.24-py3-none-any.whl') as archive:
    archive.extractall(VENDOR)
sys.path.insert(0,str(VENDOR))
sys.path.insert(0,str(ASSETS))

# Keep Kaggle Torch/CUDA and NumPy. Install missing runtime dependencies only.
import importlib.util, importlib.metadata, subprocess
dependencies = {'cv2':'opencv-python','PIL':'pillow','yaml':'pyyaml','numpy':'numpy',
    'scipy':'scipy','matplotlib':'matplotlib','polars':'polars','psutil':'psutil',
    'requests':'requests','thop':'ultralytics-thop'}
missing = [package for module,package in dependencies.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install','--quiet',*missing])
try:
    coco_version = importlib.metadata.version('pycocotools')
except importlib.metadata.PackageNotFoundError:
    coco_version = None
if coco_version != '2.0.11':
    subprocess.check_call([sys.executable,'-m','pip','install','--quiet','--no-deps','pycocotools==2.0.11'])

import torch, ultralytics
from ultralytics import settings
assert ultralytics.__version__=='8.4.24'
assert torch.cuda.is_available(), 'Enable GPU in Kaggle settings'
settings.update({'sync':False,'wandb':False,'comet':False,'mlflow':False,'clearml':False,'neptune':False,
    'dvc':False,'hub':False,'tensorboard':False})

expected = json.loads((ASSETS/'expected-runtime.json').read_text())
actual = {k:importlib.metadata.version(k) for k in expected}
assert actual == expected, f'Runtime differs from the previous recipe: expected={expected}, actual={actual}. Return this error for inspection.'
assert torch.cuda.device_count()==2 and all('T4' in torch.cuda.get_device_name(i) for i in range(2)), 'Select GPU T4 x2.'
import data_runtime, suite
print('Verified runtime:',actual)


In [ ]:
preparation = data_runtime.prepare(RUN_ROOT,ASSETS,DATASET_ROOT)
print(json.dumps(preparation,indent=2))
protocol = json.loads((ASSETS/'protocol.json').read_text())
print('Primary metrics:',protocol['primary'])
print('60 epochs each; batch8 each; AdamW lr0=.001; workers2; AMP; no early stop/resume/batch fallback.')


Each worker has its own settings, label cache, RNG, optimizer and output folder.
The two T4 memories remain separate. If either seed fails, the other is allowed to finish.
Evaluation-mode preflight checks batch8/640 forward; actual training memory fit remains unverified.
The first completed epoch reports timing and peak allocated memory; no reliable total-time estimate exists yet.


In [ ]:
from IPython.display import FileLink, display
try:
    results = suite.run_all(RUN_ROOT,ASSETS,VENDOR)
finally:
    archive = RUN_ROOT.parent / (RUN_ROOT.name+'_results.zip')
    if archive.exists():
        display(FileLink(str(archive)))


In [ ]:
import pandas as pd
display(pd.read_csv(RUN_ROOT/'review-table.csv'))
print('AP columns use 0–1. Multiply by 100 for percentages.')
display(FileLink(str(RUN_ROOT.parent/(RUN_ROOT.name+'_results.zip'))))


## Results

The archive includes best/last/periodic checkpoints, exact settings, runtime and initialization audits,
per-epoch logs, raw validation predictions, overall metrics, tiny/small-fire AP and small-smoke AP.
Size AP uses proportional fit to 640, sqrt(box area)<16/<32; small includes tiny.
Combined and per-source validation metrics are saved. COCO AP uses maxDets100/image/category.
Operating diagnostics report recall at the 30 false-fire-boxes/100-images ceiling, and false boxes at 90% tiny recall.
Fire-free image alarm counts are separate. These are still-image diagnostics, not video false-alarm/onset results.

`best.pt` follows standard overall validation fitness. The test split and the reserved 20 videos are untouched.
Both seeds share the saved pre-training backbone/head state; neck initialization and training randomness vary.
Two seeds and a descriptive mean are reported; no significance or architecture-gain claim is implied.
Custom checkpoints require the bundled `dense_model.py` importable with Ultralytics8.4.24.
To repeat evaluation without training, use `runner.py <saved configuration.json> --evaluate-only`
in the same prepared runtime and one-GPU environment. No local training was performed to prepare this notebook.
